# Bem-te-viz — preparação integral do pacote de aves

O notebook usa os datasets já anexados em `/kaggle/input/datasets`; não baixa nada pelo KaggleHub e não carrega os binários de todas as mídias para a memória. A configuração exporta todas as imagens do CUB-200-2011 (11.788 no conjunto atual) e, para cada espécie, no máximo uma gravação de áudio selecionada deterministicamente e limitada por tamanho. Isso ultrapassa 10.000 mídias sem copiar os ~31 GB de áudio.

A chave primária de relação do áudio é `class_id`, documentada como alinhada à ordem do CUB; `file_path` ou `xc_id` localiza o arquivo e o nome da espécie é validado. Conflitos ficam no relatório para revisão.

`ALLOW_MEDIA_EXPORT` começa `False`. Depois de conferir as licenças, altere para `True` e rode **Run All**. O notebook valida e monta um ZIP em `/kaggle/working`; extraia `bem-te-viz-package.zip` em `data/processed/` no projeto Node. A demonstração local permanece pequena e funciona se o pacote completo não estiver presente.


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import csv, json, os, re, shutil, tempfile, unicodedata, zipfile

INPUT_ROOT = Path('/kaggle/input/datasets')
OUTPUT_ROOT = Path('/kaggle/working')
PUBLISHERS = {'images': 'wenewone', 'audio': 'gevorgalaverdyan'}
DEMO_SPECIES_LIMIT = None  # None = exportar todas as espécies com imagens
MAX_IMAGES_PER_SPECIES = None  # None = exportar todas as imagens do CUB-200
MAX_AUDIOS_PER_SPECIES = 1  # uma gravação por espécie já basta para multimídia
# Só altere para True depois de conferir os termos de redistribuição de cada dataset.
ALLOW_MEDIA_EXPORT = False
IMAGE_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.webp'}
AUDIO_EXTENSIONS = {'.wav', '.mp3', '.ogg', '.flac', '.m4a'}
MAX_AUDIO_BYTES = 2_000_000  # escolhe a menor gravação válida até este tamanho
PACKAGE_NAME = 'bem-te-viz-package'


def normalize_key(value):
    value = unicodedata.normalize('NFKD', str(value or ''))
    value = ''.join(ch for ch in value if not unicodedata.combining(ch))
    value = value.casefold().replace('_', ' ').replace('.', ' ')
    return ' '.join(value.split())


def walk_limited(root, max_depth=2):
    root = Path(root)
    if not root.exists():
        return
    base_depth = len(root.parts)
    for current, dirs, files in os.walk(root):
        depth = len(Path(current).parts) - base_depth
        dirs.sort(); files.sort()
        if depth >= max_depth:
            dirs[:] = []
        yield Path(current), dirs, files


def print_structure(root, label):
    root = Path(root)
    print(f'\n[{label}] {root} | existe={root.exists()}')
    if not root.exists():
        return
    for current, dirs, files in walk_limited(root, max_depth=2):
        rel = current.relative_to(root) if current != root else Path('.')
        print(f'  {rel}: {len(dirs)} pastas, {len(files)} arquivos')
        if files:
            examples = files[:5]
            print('    exemplos:', ', '.join(examples))
        interesting = [f for f in files if Path(f).suffix.lower() in {'.txt', '.csv', '.tsv', '.json', '.xml'}]
        if interesting:
            print('    metadados:', ', '.join(interesting[:12]))


def find_cub_root(search_root):
    candidates = []
    for current, dirs, files in os.walk(search_root):
        dirs.sort()
        have = set(files)
        if {'classes.txt', 'images.txt', 'image_class_labels.txt'} <= have:
            candidates.append(Path(current))
    if not candidates:
        raise FileNotFoundError('Não achei classes.txt, images.txt e image_class_labels.txt juntos no dataset de imagens.')
    return sorted(candidates, key=lambda p: (len(p.parts), str(p)))[0]


def discover_audio_files(audio_root):
    found = []
    metadata = []
    for current, dirs, files in os.walk(audio_root):
        dirs.sort(); files.sort()
        for name in files:
            p = Path(current) / name
            ext = p.suffix.lower()
            if ext in AUDIO_EXTENSIONS or ext in {'.aiff', '.aac', '.wma', '.mp4'}:
                found.append(p)
            elif ext in {'.csv', '.tsv', '.json', '.txt', '.xml'}:
                try:
                    if p.stat().st_size <= 8 * 1024 * 1024:
                        metadata.append((p, p.stat().st_size))
                except OSError:
                    pass
    return found, metadata


def load_cub_metadata(cub_root):
    classes = {}
    with (cub_root / 'classes.txt').open(encoding='utf-8') as f:
        for line in f:
            parts = line.strip().split(maxsplit=1)
            if len(parts) == 2:
                class_id, raw_label = parts
                label = raw_label.split('.', 1)[-1]
                classes[class_id] = {'raw': raw_label, 'label': label, 'key': normalize_key(label)}
    image_names = {}
    with (cub_root / 'images.txt').open(encoding='utf-8') as f:
        for line in f:
            parts = line.strip().split(maxsplit=1)
            if len(parts) == 2:
                image_names[parts[0]] = parts[1]
    image_classes = {}
    with (cub_root / 'image_class_labels.txt').open(encoding='utf-8') as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) >= 2:
                image_classes[parts[0]] = parts[1]
    records = {}
    for class_id, cls in sorted(classes.items(), key=lambda item: int(item[0])):
        records[class_id] = {
            'id': class_id,
            'species': cls['label'].replace('_', ' '),
            'originalLabel': cls['raw'],
            'key': cls['key'],
            'images': [],
            'audios': [],
            'errors': [],
        }
    for image_id, relative in sorted(image_names.items(), key=lambda item: int(item[0])):
        class_id = image_classes.get(image_id)
        if class_id in records:
            records[class_id]['images'].append(cub_root / 'images' / relative)
    return records


def exact_audio_matches(records, audio_files, audio_root, metadata_files):
    # CUB e o dataset de áudio compartilham class_id (1..200). Esse ID é a chave
    # primária; file_path/xc_id identificam o arquivo. O nome serve como auditoria.
    by_key = {record['key']: class_id for class_id, record in records.items()}
    by_numeric_id = {str(int(class_id)): class_id for class_id in records}
    audio_by_name, audio_by_rel, audio_by_xc = {}, {}, {}
    for audio_path in audio_files:
        audio_by_name.setdefault(normalize_key(audio_path.name), []).append(audio_path)
        audio_by_name.setdefault(normalize_key(audio_path.stem), []).append(audio_path)
        rel = audio_path.relative_to(audio_root)
        audio_by_rel[normalize_key(rel.as_posix())] = audio_path
        parts = rel.parts
        # O metadata.csv usa caminho relativo a data/, enquanto a raiz anexada pode
        # incluir essa pasta como primeiro componente.
        if parts and parts[0].casefold() in {'data', 'audio', 'audios'} and len(parts) > 1:
            audio_by_rel[normalize_key(Path(*parts[1:]).as_posix())] = audio_path
        match = re.search(r'_(\d+)$', audio_path.stem)
        if match:
            audio_by_xc.setdefault(match.group(1), []).append(audio_path)

    metadata_links, metadata_methods = {}, {}
    metadata_conflicts, metadata_errors = [], []
    species_columns = {'species', 'species name', 'common name', 'class name', 'label', 'bird name', 'taxon name', 'scientific name', 'scientific species name', 'binomial', 'latin name'}
    audio_columns = {'filename', 'file name', 'audio file', 'audio filename', 'audio file path', 'filepath', 'file path', 'path', 'recording'}
    id_columns = {'class id', 'classid', 'species id', 'label id'}
    xc_columns = {'xc id', 'xcid', 'recording id'}
    for metadata_path, _ in metadata_files:
        if metadata_path.suffix.lower() not in {'.csv', '.tsv'}:
            continue
        try:
            with metadata_path.open(encoding='utf-8-sig', newline='') as f:
                reader = csv.DictReader(f, delimiter='\t' if metadata_path.suffix.lower() == '.tsv' else ',')
                if not reader.fieldnames:
                    continue
                normalized_columns = {normalize_key(name): name for name in reader.fieldnames if name}
                species_col = next((normalized_columns[name] for name in species_columns if name in normalized_columns), None)
                audio_col = next((normalized_columns[name] for name in audio_columns if name in normalized_columns), None)
                id_col = next((normalized_columns[name] for name in id_columns if name in normalized_columns), None)
                xc_col = next((normalized_columns[name] for name in xc_columns if name in normalized_columns), None)
                if not audio_col:
                    continue
                for row in reader:
                    raw_audio = (row.get(audio_col) or '').strip().replace('\\', '/')
                    if not raw_audio:
                        continue
                    metadata_key = normalize_key(raw_audio)
                    candidate = audio_by_rel.get(metadata_key)
                    candidates = [candidate] if candidate else []
                    if not candidates and xc_col:
                        xc_id = re.sub(r'\D', '', str(row.get(xc_col) or ''))
                        if xc_id:
                            candidates = audio_by_xc.get(xc_id, [])
                    if not candidates:
                        basename = Path(raw_audio).name
                        candidates = audio_by_name.get(normalize_key(basename), [])
                        if not candidates and Path(basename).suffix:
                            candidates = audio_by_name.get(normalize_key(Path(basename).stem), [])
                    candidates = list(dict.fromkeys(candidates))
                    class_id = None
                    if id_col:
                        id_digits = re.sub(r'\D', '', str(row.get(id_col) or ''))
                        if id_digits:
                            try: class_id = by_numeric_id.get(str(int(id_digits)))
                            except ValueError: pass
                    # Fallback exato para metadado sem class_id: nome + arquivo.
                    if class_id is None and species_col:
                        class_id = by_key.get(normalize_key(row.get(species_col, '')))
                    if class_id and len(candidates) == 1:
                        audio_path = candidates[0]
                        # Se o diretório explicita um ID, ele não pode contradizer o CSV.
                        folder_ids = [re.match(r'0*(\d+)\.', part) for part in audio_path.relative_to(audio_root).parts[:-1]]
                        folder_ids = [str(int(m.group(1))) for m in folder_ids if m]
                        if folder_ids and str(int(class_id)) not in folder_ids:
                            metadata_errors.append({'path': str(audio_path), 'error': f'class_id {class_id} diverge do ID da pasta {folder_ids}'})
                            continue
                        metadata_links.setdefault(audio_path, set()).add(class_id)
                        method = 'class_id+file_path' if id_col and metadata_key in audio_by_rel else ('class_id+xc_id' if id_col and xc_col else 'metadado_exato')
                        metadata_methods[audio_path] = method
                        if species_col and normalize_key(row.get(species_col, '')) != records[class_id]['key']:
                            metadata_conflicts.append({'audio': audio_path.relative_to(audio_root).as_posix(), 'class_id': class_id, 'catalogSpecies': records[class_id]['species'], 'metadataSpecies': row.get(species_col, '')})
        except Exception as exc:
            metadata_errors.append({'path': str(metadata_path), 'error': f'falha ao ler metadados: {exc}'})

    matches, unmatched, errors = [], [], list(metadata_errors)
    for audio_path in sorted(audio_files, key=lambda p: str(p).casefold()):
        try:
            info = audio_path.stat()
        except OSError as exc:
            errors.append({'path': str(audio_path), 'error': f'stat: {exc}'})
            continue
        if info.st_size == 0:
            errors.append({'path': str(audio_path), 'error': 'arquivo vazio'})
            continue
        if audio_path.suffix.lower() not in AUDIO_EXTENSIONS:
            errors.append({'path': str(audio_path), 'error': f'formato não suportado: {audio_path.suffix}'})
            continue
        rel = audio_path.relative_to(audio_root)
        ids = metadata_links.get(audio_path, set())
        match_method = metadata_methods.get(audio_path, 'rotulo_exato_no_caminho')
        if not ids:
            # Último recurso estrito: compara componentes de pasta e rótulo completo.
            candidates = [normalize_key(part) for part in rel.parts[:-1]]
            candidates.append(normalize_key(audio_path.stem))
            ids = {by_key[key] for key in candidates if key in by_key}
        if len(ids) == 1:
            class_id = next(iter(ids))
            records[class_id]['audios'].append(audio_path)
            matches.append({'speciesId': class_id, 'species': records[class_id]['species'], 'audio': rel.as_posix(), 'matchKey': 'class_id' if match_method.startswith('class_id') else records[class_id]['key'], 'method': match_method})
        elif len(ids) > 1:
            errors.append({'path': rel.as_posix(), 'error': f'chave ambígua: {sorted(ids, key=int)}'})
        else:
            unmatched.append(rel.as_posix())
    return matches, unmatched, errors, metadata_conflicts

def inspect_metadata(metadata_files):
    print('\nMetadados localizados no dataset de áudio (nomes e primeiras linhas; limite 8 MiB por arquivo):')
    for p, size in metadata_files[:30]:
        print(f'  {p} ({size:,} bytes)')
        if p.suffix.lower() in {'.csv', '.tsv'}:
            try:
                with p.open(encoding='utf-8-sig', newline='') as f:
                    dialect = csv.excel_tab if p.suffix.lower() == '.tsv' else csv.excel
                    reader = csv.reader(f, dialect=dialect)
                    for _, row in zip(range(3), reader):
                        print('    ', row[:12])
            except Exception as exc:
                print('    não foi possível ler amostra:', exc)
        elif p.suffix.lower() == '.txt':
            try:
                with p.open(encoding='utf-8', errors='replace') as f:
                    for _, line in zip(range(3), f): print('    ', line.rstrip()[:200])
            except OSError as exc:
                print('    não foi possível ler amostra:', exc)


def check_source_files(records):
    errors, seen = [], set()
    for record in records.values():
        for field, supported in [('images', IMAGE_EXTENSIONS), ('audios', AUDIO_EXTENSIONS)]:
            unique = []
            for p in sorted(record[field], key=lambda item: str(item).casefold()):
                resolved = str(p.resolve())
                if resolved in seen:
                    errors.append({'path': resolved, 'error': 'referência repetida no índice de origem'})
                    continue
                seen.add(resolved)
                try:
                    size = p.stat().st_size
                except OSError as exc:
                    errors.append({'path': resolved, 'error': f'caminho inexistente: {exc}'})
                    continue
                if size == 0:
                    errors.append({'path': resolved, 'error': 'arquivo vazio'})
                    continue
                if p.suffix.lower() not in supported:
                    errors.append({'path': resolved, 'error': f'formato não suportado: {p.suffix}'})
                    continue
                unique.append(p)
            record[field] = unique
    return errors


def copy_unique(source, destination, prefix):
    destination.parent.mkdir(parents=True, exist_ok=True)
    final = destination.with_name(f'{prefix}_{destination.name}')
    shutil.copyfile(source, final)
    return final


def validate_package(root):
    catalog = json.loads((root / 'catalog.json').read_text(encoding='utf-8'))
    refs, errors = set(), []
    for bird in catalog['birds']:
        for field, supported in [('images', IMAGE_EXTENSIONS), ('audios', AUDIO_EXTENSIONS)]:
            for rel in bird[field]:
                if rel in refs:
                    errors.append(f'referência duplicada: {rel}')
                refs.add(rel)
                p = (root / rel).resolve()
                if root.resolve() not in p.parents:
                    errors.append(f'caminho fora do pacote: {rel}')
                elif not p.is_file():
                    errors.append(f'caminho inexistente: {rel}')
                elif p.stat().st_size == 0:
                    errors.append(f'arquivo vazio: {rel}')
                elif p.suffix.lower() not in supported:
                    errors.append(f'formato não suportado: {rel}')
    return errors


def collect_credits(roots):
    credit_docs = []
    needles = ('license', 'licence', 'readme', 'citation', 'dataset-metadata')
    for root in roots:
        root = Path(root)
        if not root.exists():
            continue
        for current, dirs, files in os.walk(root):
            dirs.sort(); files.sort()
            for filename in files:
                if not any(word in filename.casefold() for word in needles):
                    continue
                p = Path(current) / filename
                try:
                    size = p.stat().st_size
                    item = {'path': str(p), 'sizeBytes': size, 'excerpt': None}
                    if size <= 64 * 1024 and p.suffix.lower() in {'.txt', '.md', '.json', '.yaml', '.yml'}:
                        item['excerpt'] = p.read_text(encoding='utf-8', errors='replace')[:4000]
                    credit_docs.append(item)
                except OSError as exc:
                    credit_docs.append({'path': str(p), 'error': str(exc)})
    return credit_docs


def run_synthetic_tests():
    assert normalize_key('  Guarubá__Guarouba ') == 'guaruba guarouba'
    records = {
        '1': {'id': '1', 'species': 'Test Bird', 'key': normalize_key('Test_Bird'), 'images': [], 'audios': [], 'errors': []},
        '2': {'id': '2', 'species': 'Another Bird', 'key': normalize_key('Another_Bird'), 'images': [], 'audios': [], 'errors': []},
    }
    with tempfile.TemporaryDirectory() as tmp:
        tmp = Path(tmp)
        audio_root = tmp / 'audio'; audio_root.mkdir()
        class_dir = audio_root / 'data' / '001.Test_Bird'; class_dir.mkdir(parents=True)
        first = class_dir / 'Test_Bird_100.mp3'; first.write_bytes(b'audio-1')
        second = class_dir / 'Test_Bird_101.mp3'; second.write_bytes(b'audio-2')
        orphan_dir = audio_root / 'data' / '999.Unknown_Bird'; orphan_dir.mkdir(parents=True)
        orphan = orphan_dir / 'Unknown_Bird_900.mp3'; orphan.write_bytes(b'audio-orphan')
        metadata = tmp / 'metadata.csv'
        with metadata.open('w', newline='', encoding='utf-8') as f:
            writer = csv.DictWriter(f, fieldnames=['file_path', 'class_id', 'species', 'xc_id'])
            writer.writeheader()
            writer.writerow({'file_path': '001.Test_Bird/Test_Bird_100.mp3', 'class_id': '1', 'species': 'Test Bird', 'xc_id': '100'})
            writer.writerow({'file_path': '001.Test_Bird/Test_Bird_101.mp3', 'class_id': '1', 'species': 'Test Bird', 'xc_id': '101'})
            writer.writerow({'file_path': '999.Unknown_Bird/Unknown_Bird_900.mp3', 'class_id': '999', 'species': 'Unknown Bird', 'xc_id': '900'})
        audio_files = sorted(audio_root.rglob('*.mp3'))
        matches, unmatched, errors, conflicts = exact_audio_matches(
            records, audio_files, audio_root, [(metadata, metadata.stat().st_size)]
        )
        assert len(matches) == 2 and len(records['1']['audios']) == 2  # mesma espécie, várias gravações
        assert len(unmatched) == 1 and orphan.relative_to(audio_root).as_posix() in unmatched  # sem par
        assert not conflicts
        package = tmp / 'package'; (package / 'media' / 'images').mkdir(parents=True)
        (package / 'catalog.json').write_text(json.dumps({'birds': [{'id': '1', 'species': 'Test Bird', 'images': ['media/images/missing.jpg'], 'audios': []}]}))
        (package / 'manifest.json').write_text(json.dumps({'validation': {'status': 'passed'}}))
        package_errors = validate_package(package)
        assert any('caminho inexistente' in error for error in package_errors)  # arquivo ausente
    print('Testes sintéticos: class_id/file_path, espécie sem par, várias mídias e caminho inexistente: OK')

def main():
    if not INPUT_ROOT.exists():
        raise FileNotFoundError(f'Entrada não encontrada: {INPUT_ROOT}. Anexe os dois datasets ao notebook.')
    image_root = INPUT_ROOT / PUBLISHERS['images']
    audio_root = INPUT_ROOT / PUBLISHERS['audio']
    print_structure(image_root, 'CUB-200-2011 (imagens)')
    print_structure(audio_root, 'dataset de áudio')
    cub_root = find_cub_root(image_root)
    print(f'\nRaiz CUB detectada: {cub_root}')
    records = load_cub_metadata(cub_root)
    audio_files, metadata_files = discover_audio_files(audio_root)
    inspect_metadata(metadata_files)
    credits = collect_credits([image_root, audio_root])
    print(f'\nArquivos de áudio encontrados por extensão: {len(audio_files):,}; metadados inspecionados: {len(metadata_files):,}')
    matches, unmatched_audio, audio_errors, metadata_conflicts = exact_audio_matches(records, audio_files, audio_root, metadata_files)
    source_errors = check_source_files(records)
    all_errors = source_errors + audio_errors
    for record in records.values():
        record['images'].sort(key=lambda p: str(p).casefold())
        record['audios'].sort(key=lambda p: str(p).casefold())

    # Demonstração determinística: até N espécies com imagem; limita mídias por espécie.
    eligible = [cid for cid, r in sorted(records.items(), key=lambda item: int(item[0])) if r['images']]
    selected_ids = set(eligible if DEMO_SPECIES_LIMIT is None else eligible[:DEMO_SPECIES_LIMIT])
    stage = OUTPUT_ROOT / f'.{PACKAGE_NAME}.tmp'
    target = OUTPUT_ROOT / PACKAGE_NAME
    backup = OUTPUT_ROOT / f'.{PACKAGE_NAME}.previous'
    if not target.exists() and backup.exists():
        backup.rename(target)  # recupera o pacote válido se uma execução anterior foi interrompida durante a troca
    if stage.exists():
        shutil.rmtree(stage)
    stage.mkdir(parents=True)
    (stage / 'media' / 'images').mkdir(parents=True)
    (stage / 'media' / 'audios').mkdir(parents=True)

    app_birds, metadata_birds, included_images, included_audios = [], [], 0, 0
    for cid, r in sorted(records.items(), key=lambda item: int(item[0])):
        metadata_birds.append({'id': r['id'], 'species': r['species'], 'originalLabel': r['originalLabel'], 'imageCount': len(r['images']), 'audioCount': len(r['audios'])})
        out = {'id': r['id'], 'species': r['species'], 'originalLabel': r['originalLabel'], 'images': [], 'audios': []}
        if cid in selected_ids and ALLOW_MEDIA_EXPORT:
            image_candidates = r['images'] if MAX_IMAGES_PER_SPECIES is None else r['images'][:MAX_IMAGES_PER_SPECIES]
            for i, src in enumerate(image_candidates, start=1):
                rel = Path('media/images') / f'{cid}_{i}_{src.name}'
                copied = stage / rel
                shutil.copyfile(src, copied)
                out['images'].append(rel.as_posix()); included_images += 1
            eligible_audio = []
            for src in r['audios']:
                try:
                    size = src.stat().st_size
                except OSError:
                    continue
                if 0 < size <= MAX_AUDIO_BYTES:
                    eligible_audio.append((size, str(src).casefold(), src))
            eligible_audio.sort(key=lambda item: (item[0], item[1]))
            audio_candidates = [item[2] for item in eligible_audio]
            if MAX_AUDIOS_PER_SPECIES is not None:
                audio_candidates = audio_candidates[:MAX_AUDIOS_PER_SPECIES]
            for i, src in enumerate(audio_candidates, start=1):
                rel = Path('media/audios') / f'{cid}_{i}_{src.name}'
                copied = stage / rel
                shutil.copyfile(src, copied)
                out['audios'].append(rel.as_posix()); included_audios += 1
        app_birds.append(out)

    catalog = {'schemaVersion': 1, 'birds': app_birds}
    (stage / 'catalog.json').write_text(json.dumps(catalog, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
    (stage / 'species-metadata.json').write_text(json.dumps({'species': metadata_birds}, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
    package_errors = validate_package(stage)
    if ALLOW_MEDIA_EXPORT and included_images + included_audios < 10000:
        package_errors.append(f'Pacote contém menos de 10000 mídias: {included_images + included_audios}')
    report = {
        'speciesCount': len(records), 'audioFilesFound': len(audio_files), 'matchedAudioFiles': len(matches),
        'unmatchedAudioFiles': len(unmatched_audio), 'speciesWithAudio': sum(bool(r['audios']) for r in records.values()),
        'speciesWithoutAudioMatch': [r['species'] for r in records.values() if not r['audios']],
        'speciesSelectedForDemo': len(selected_ids), 'mediaExportEnabled': ALLOW_MEDIA_EXPORT, 'sourceImageCount': sum(len(r['images']) for r in records.values()), 'sourceAudioCount': len(audio_files), 'matchedAudioCount': len(matches), 'mediaIncludedCount': included_images + included_audios, 'includedImages': included_images, 'includedAudios': included_audios, 'maxAudioBytes': MAX_AUDIO_BYTES, 'metadataNameConflicts': metadata_conflicts,
        'matches': matches, 'unmatchedAudioExamples': unmatched_audio[:100],
        'sourceErrors': all_errors[:500], 'packageErrors': package_errors,
        'sourceCreditDocuments': credits,
        'limitations': ['A associação prioritária usa class_id do CSV e file_path/xc_id exatos; o rótulo é auditado e conflitos são reportados.', 'Arquivos sem ID ou caminho exato permanecem sem correspondência; nenhuma aproximação fonética ou textual é usada.', 'A exportação de bytes fica desligada até alguém conferir os termos dos dois datasets e alterar ALLOW_MEDIA_EXPORT.', 'O catálogo metadata não inclui bytes de mídia; o catálogo de demonstração inclui apenas a amostra limitada.'],
    }
    credits_text = ['# Créditos e licenças', '', 'Fontes: Kaggle datasets `wenewone/cub2002011` (CUB-200-2011) e `gevorgalaverdyan/cub-200-bird-audio-dataset`. Confira os termos originais antes de redistribuir mídias.', '', 'Documentos de licença/citação encontrados:']
    credits_text += [f"- `{Path(item['path']).name}` em `{item['path']}` ({item.get('sizeBytes', 0)} bytes)" for item in credits]
    if not credits: credits_text.append('- Nenhum arquivo de licença/citação foi encontrado nas pastas anexadas; licença não confirmada pelo notebook.')
    credits_text += ['', f'Mídias copiadas: {ALLOW_MEDIA_EXPORT}. A configuração começa desligada. Só habilite depois de confirmar os termos de redistribuição dos dois datasets no Kaggle.']
    (stage / 'CREDITS.md').write_text('\n'.join(credits_text) + '\n', encoding='utf-8')
    (stage / 'report.json').write_text(json.dumps(report, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
    validation = 'passed' if not package_errors else 'failed'
    manifest = {
        'generatedAt': datetime.now(timezone.utc).isoformat(),
        'sources': {'images': str(image_root), 'audio': str(audio_root), 'cubRoot': str(cub_root), 'publishers': PUBLISHERS},
        'configuration': {'demoSpeciesLimit': DEMO_SPECIES_LIMIT, 'maxImagesPerSpecies': MAX_IMAGES_PER_SPECIES, 'maxAudiosPerSpecies': MAX_AUDIOS_PER_SPECIES, 'maxAudioBytes': MAX_AUDIO_BYTES, 'allowMediaExport': ALLOW_MEDIA_EXPORT, 'imageExtensions': sorted(IMAGE_EXTENSIONS), 'audioExtensions': sorted(AUDIO_EXTENSIONS)},
        'speciesCount': len(records), 'includedImages': included_images, 'includedAudios': included_audios,
        'validation': {'status': validation, 'errors': package_errors},
        'matchKey': 'class_id CUB compartilhado + file_path/xc_id exatos; validação do rótulo da espécie; fallback por igualdade exata do rótulo completo',
    }
    (stage / 'manifest.json').write_text(json.dumps(manifest, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
    if package_errors:
        raise RuntimeError(f'Pacote não publicado; erros de validação: {package_errors[:10]}')

    # Publicação com diretório temporário; pacote anterior fica preservado até nova validação.
    if backup.exists(): shutil.rmtree(backup)
    if target.exists(): target.rename(backup)
    try:
        stage.rename(target)
    except Exception:
        if backup.exists() and not target.exists(): backup.rename(target)
        raise
    if backup.exists(): shutil.rmtree(backup)

    zip_tmp = OUTPUT_ROOT / f'.{PACKAGE_NAME}.zip.tmp'
    zip_final = OUTPUT_ROOT / f'{PACKAGE_NAME}.zip'
    with zipfile.ZipFile(zip_tmp, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
        for p in sorted(target.rglob('*')):
            if p.is_file(): zf.write(p, p.relative_to(OUTPUT_ROOT))
    zip_tmp.replace(zip_final)
    print('\nPacote gerado:', target)
    print('ZIP para baixar no painel Output do Kaggle:', zip_final, f'({zip_final.stat().st_size:,} bytes)')
    print(f"Espécies: {len(records)} | áudios relacionados: {len(matches)} | áudios sem par: {len(unmatched_audio)} | validação: {validation}")
    print('Para usar no Node: extraia o ZIP em data/processed/; o catálogo aponta apenas para as mídias copiadas.')
    return report

run_synthetic_tests()
report = main()
